In [16]:
# https://chatgpt.com/c/69b68ac2-0f48-8325-a819-24c43bef79cc
import kagglehub
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

# =========================
# Download dataset
# =========================
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Path to dataset files:", path)

dataset_path = Path(path) / "chest_xray"
print(dataset_path)

merged_dir = Path("merged_dataset")
output_dir = Path("dataset_70_30")

classes = ["NORMAL", "PNEUMONIA"]
splits = ["train", "val", "test"]

# =========================
# Clean old folders
# =========================
if merged_dir.exists():
    shutil.rmtree(merged_dir)

if output_dir.exists():
    shutil.rmtree(output_dir)

print("Old folders removed")

# =========================
# Create merged folders
# =========================
for c in classes:
    (merged_dir / c).mkdir(parents=True, exist_ok=True)

# =========================
# Merge original dataset
# =========================
counter = 0

for split in splits:
    for c in classes:
        source = dataset_path / split / c

        if not source.exists():
            continue

        for img in source.iterdir():

            if img.suffix.lower() not in [".jpeg", ".jpg", ".png"]:
                continue

            new_name = f"{split}_{counter}_{img.name}"
            shutil.copy(img, merged_dir / c / new_name)
            counter += 1

print(f"Merging completed. Total images copied: {counter}")

# =========================
# Create 70 / 30 split
# =========================
train_dir = output_dir / "train"
test_dir = output_dir / "test"

all_images = []
all_labels = []

for c in classes:
    class_folder = merged_dir / c

    for img in class_folder.iterdir():
        if img.suffix.lower() in [".jpeg", ".jpg", ".png"]:
            all_images.append(img)
            all_labels.append(c)

print("Total merged images:", len(all_images))

train_imgs, test_imgs, train_labels, test_labels = train_test_split(
    all_images,
    all_labels,
    test_size=0.30,
    random_state=42,
    stratify=all_labels
)

print("Train images:", len(train_imgs))
print("Test images:", len(test_imgs))

# =========================
# Create split folders
# =========================
for c in classes:
    (train_dir / c).mkdir(parents=True, exist_ok=True)
    (test_dir / c).mkdir(parents=True, exist_ok=True)

# Copy training images
for img, label in zip(train_imgs, train_labels):
    shutil.copy(img, train_dir / label / img.name)

# Copy test images
for img, label in zip(test_imgs, test_labels):
    shutil.copy(img, test_dir / label / img.name)

print("70/30 split created successfully")
print("Training data saved in:", train_dir)
print("Test data saved in:", test_dir)



import tensorflow as tf
import numpy as np
from sklearn.metrics import accuracy_score

train_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset_70_30/train", image_size=(180,180), batch_size=32, color_mode="grayscale", shuffle=False
).map(lambda x, y: (x / 255.0, y))

test_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset_70_30/test", image_size=(180,180), batch_size=32, color_mode="grayscale", shuffle=False
).map(lambda x, y: (x / 255.0, y))

model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(30, 3, activation='relu', input_shape=(180,180,1)),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(30, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(train_ds, validation_data=test_ds, epochs=20)

y_train = np.concatenate([y.numpy() for x, y in train_ds])
y_test = np.concatenate([y.numpy() for x, y in test_ds])

pred_train = (model.predict(train_ds) > 0.5).astype(int).flatten()
pred_test = (model.predict(test_ds) > 0.5).astype(int).flatten()

print(f"Training Accuracy: {accuracy_score(y_train, pred_train):.3f}")
print(f"Test Accuracy: {accuracy_score(y_test, pred_test):.3f}")

# =========================
# CleanUp
# =========================
if merged_dir.exists():
    shutil.rmtree(merged_dir)

if output_dir.exists():
    shutil.rmtree(output_dir)
print("Cleanup is done")

Path to dataset files: /Users/dineshbisht/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2
/Users/dineshbisht/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray
Old folders removed
Merging completed. Total images copied: 5856
Total merged images: 5856
Train images: 4099
Test images: 1757
70/30 split created successfully
Training data saved in: dataset_70_30/train
Test data saved in: dataset_70_30/test
Found 4099 files belonging to 2 classes.
Found 1757 files belonging to 2 classes.
Epoch 1/20


/Users/dineshbisht/masterdegree/myproject_env/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


129/129 ━━━━━━━━━━━━━━━━━━━━ 10s 75ms/step - accuracy: 0.9546 - loss: 1.3088 - val_accuracy: 0.7297 - val_loss: 4.9317
Epoch 2/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 9s 73ms/step - accuracy: 0.6350 - loss: 1.1707 - val_accuracy: 0.7297 - val_loss: 14.4569
Epoch 3/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 9s 72ms/step - accuracy: 0.8607 - loss: 2.5865 - val_accuracy: 0.7297 - val_loss: 13.6794
Epoch 4/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 9s 71ms/step - accuracy: 0.5209 - loss: 2.1667 - val_accuracy: 0.7297 - val_loss: 0.6809
Epoch 5/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 9s 71ms/step - accuracy: 0.7297 - loss: 0.6757 - val_accuracy: 0.7297 - val_loss: 0.6675
Epoch 6/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 10s 73ms/step - accuracy: 0.7297 - loss: 0.6634 - val_accuracy: 0.7297 - val_loss: 0.6563
Epoch 7/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 776s 6s/step - accuracy: 0.7297 - loss: 0.6531 - val_accuracy: 0.7297 - val_loss: 0.6468
Epoch 8/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 1808s 14s/step - accuracy: 0.7297 - loss: 0.6443 - val_accuracy: